# Probability calibration

The frozen 80% LightGBM / 20% XGBoost champion ranks fraud well, but its raw scores require a reliability check. Calibration maps scores to observed fraud probability. It is not intended to improve ranking PR-AUC. No features, model parameters, weights, or thresholds are changed.


In [1]:
import gc
import json
import sys
import time
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from fraud_detection.data import merge_train_data
from fraud_detection.features import COMPOSITE_TARGET_ENTITIES
from fraud_detection.models import prepare_engineered_matrices
from fraud_detection.models.calibration import (
    Calibrator,
    evaluate,
    reliability_bins,
    select_method,
    temporal_selection,
    validate_oof,
)
from fraud_detection.models.ensemble import weighted_score_blend

started = time.perf_counter()

Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\Marwan\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


## TRAIN OOF source and integrity

The aligned predictions come from three chronological TRAIN folds. Calibration evaluation fits on Fold 1 and evaluates on Fold 2, then fits on Folds 1+2 and evaluates on Fold 3. The method is selected before external VALIDATION is read.


In [2]:
oof_path = ROOT / "reports/ensemble_oof_predictions.csv"
oof = pd.read_csv(oof_path)
raw_oof = validate_oof(oof)
assert len(oof) == 124014
assert oof.groupby("fold").size().tolist() == [41338, 41338, 41338]
print("OOF rows:", len(oof), "unique identifiers:", oof.TransactionID.nunique())
print("Raw OOF range:", raw_oof.min(), raw_oof.max())
print("Raw OOF all-fold metrics:", evaluate(oof.y_true, raw_oof))

OOF rows: 124014 unique identifiers: 124014
Raw OOF range: 5.2348567172217906e-06 0.9999759756761643
Raw OOF all-fold metrics: {'brier': 0.028309994664646298, 'log_loss': 0.11613464642050732, 'ece': 0.036921808526406266, 'mean_probability': 0.07455001090153997, 'prevalence': 0.038874643185446804, 'calibration_gap': 0.03567536771609316, 'pr_auc': 0.605630405687103, 'roc_auc': 0.8944614382653303}


## Temporal selection: raw, Platt, isotonic

Platt fits an unweighted logistic regression on logit(clipped raw score), with epsilon 1e-6. Isotonic fits a monotone step function and clips out-of-range future scores. Ten equal-frequency reliability bins contain every row. ECE is the sum over bins of (bin count / total count) times the absolute difference between mean probability and observed fraud rate. Pooled Brier is the primary selection metric; Log Loss and ECE are secondary. PR-AUC and ROC-AUC are sanity checks only.


In [3]:
selection, temporal_predictions = temporal_selection(oof)
display(selection)
selected = select_method(selection)
print("Selected using TRAIN OOF only:", selected)
calibrator = Calibrator(selected).fit(raw_oof, oof.y_true)
calibrator_path = ROOT / "models/probability_calibrator.joblib"
joblib.dump(calibrator, calibrator_path)
print("Saved calibrator:", calibrator_path)

C:\fraud-detection-system\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\fraud-detection-system\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,method,evaluation_fold,fit_rows,evaluation_rows,brier,log_loss,ece,mean_probability,prevalence,calibration_gap,pr_auc,roc_auc
0,raw,2,41338.0,41338,0.026363,0.107436,0.032965,0.068081,0.035222,0.032859,0.579301,0.897137
1,raw,3,82676.0,41338,0.034268,0.135178,0.046762,0.091463,0.046011,0.045452,0.631579,0.904517
2,raw,pooled,NaN,82676,0.030316,0.121307,0.039711,0.079772,0.040616,0.039155,0.608317,0.901369
3,platt,2,41338.0,41338,0.020791,0.088950,0.003287,0.037077,0.035222,0.001855,0.579301,0.897137
4,platt,3,82676.0,41338,0.024938,0.104078,0.007847,0.050771,0.046011,0.004760,0.631579,0.904517
5,platt,pooled,NaN,82676,0.022864,0.096514,0.005599,0.043924,0.040616,0.003308,0.608349,0.901322
6,isotonic,2,41338.0,41338,0.020809,0.089653,0.003648,0.037106,0.035222,0.001884,0.561445,0.896308
7,isotonic,3,82676.0,41338,0.025153,0.105260,0.006805,0.050916,0.046011,0.004905,0.617172,0.903967
8,isotonic,pooled,NaN,82676,0.022981,0.097456,0.004616,0.044011,0.040616,0.003394,0.598370,0.899432


Selected using TRAIN OOF only: platt
Saved calibrator: C:\fraud-detection-system\models\probability_calibrator.joblib


C:\fraud-detection-system\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## One external VALIDATION assessment

The original ensemble challenge did not persist validation row scores. Rebuild the exact 467-feature full-TRAIN champion and apply the already selected, all-OOF-fitted calibrator. TRAIN-only preprocessing supplies frozen target-encoding maps.


In [4]:
import io

full_rows = 590540
train_end = int(full_rows * 0.70)
validation_end = train_end + int(full_rows * 0.15)
transaction_path = ROOT / "data/raw/train_transaction.csv"
sample = pd.read_csv(transaction_path, nrows=1000, low_memory=False)
dtypes = {
    name: ("float64" if pd.api.types.is_numeric_dtype(dtype) else "object")
    for name, dtype in sample.dtypes.items()
}
dtypes["TransactionID"] = "int64"
dtypes["TransactionDT"] = "int64"
dtypes["card1"] = "int64"
dtypes["isFraud"] = "int8"
del sample
transaction = pd.read_csv(transaction_path, nrows=validation_end, dtype=dtypes)
validation_max_id = int(transaction.TransactionID.iloc[-1])
identity_path = ROOT / "data/raw/train_identity.csv"
with identity_path.open(encoding="utf-8", newline="") as handle:
    header = handle.readline()
    historical_lines = []
    for line in handle:
        if int(line.split(",", 1)[0]) > validation_max_id:
            break
        historical_lines.append(line)
identity = pd.read_csv(io.StringIO(header + "".join(historical_lines)), low_memory=False)
del historical_lines
merged = merge_train_data(transaction, identity)
train = merged.iloc[:train_end]
validation = merged.iloc[train_end:validation_end]
del transaction, identity, merged
gc.collect()
assert set(oof.TransactionID).issubset(set(train.TransactionID))
assert set(oof.TransactionID).isdisjoint(set(validation.TransactionID))
prepared = prepare_engineered_matrices(
    train,
    validation,
    include_graph=False,
    include_composite=True,
    include_target_encoding=True,
    scaling_batch_rows=len(train),
)
positions = {name: i for i, name in enumerate(prepared.feature_names)}
baseline_names = tuple(
    name for name in prepared.feature_names if not name.endswith("_target_encoded_fraud_rate")
)
composite_te_features = tuple(
    f"{entity}_target_encoded_fraud_rate" for entity in COMPOSITE_TARGET_ENTITIES
)
selected_names = baseline_names + composite_te_features
assert len(selected_names) == 467 and len(set(selected_names)) == 467
assert "isFraud" not in selected_names and "y_true" not in selected_names
indices = np.asarray([positions[name] for name in selected_names], dtype=np.int32)
x_train, x_validation = prepared.x_train[:, indices], prepared.x_evaluation[:, indices]
y_train = train.isFraud.astype("int8")
y_validation = validation.isFraud.astype("int8")
matrix_mib = (prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2
lgbm = lgb.LGBMClassifier(
    learning_rate=0.1,
    n_estimators=400,
    num_leaves=63,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
lgbm.fit(x_train, y_train)
lgbm_score = lgbm.predict_proba(x_validation)[:, 1]
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
xgboost = xgb.XGBClassifier(
    learning_rate=0.1,
    max_depth=7,
    n_estimators=300,
    tree_method="hist",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)
xgboost.fit(x_train, y_train)
xgboost_score = xgboost.predict_proba(x_validation)[:, 1]
raw_validation = weighted_score_blend(
    lgbm_score, xgboost_score, lightgbm_weight=0.8, xgboost_weight=0.2
)
calibrated_validation = calibrator.transform(raw_validation)
raw_metrics = evaluate(y_validation, raw_validation)
calibrated_metrics = evaluate(y_validation, calibrated_validation)
print("Raw reference delta PR-AUC:", raw_metrics["pr_auc"] - 0.615265)
print("Raw reference delta ROC-AUC:", raw_metrics["roc_auc"] - 0.916224)
display(pd.DataFrame([raw_metrics, calibrated_metrics], index=["raw", "calibrated"]))

Raw reference delta PR-AUC: -4.980371891560864e-07
Raw reference delta ROC-AUC: -4.3683670347682835e-08


,brier,log_loss,ece,mean_probability,prevalence,calibration_gap,pr_auc,roc_auc
raw,0.024976,0.103692,0.039006,0.073348,0.034341,0.039006,0.615265,0.916224
calibrated,0.019493,0.082039,0.005434,0.036660,0.034341,0.002319,0.615265,0.916224


## Reliability diagram and production implications

The diagonal indicates agreement between predicted probability and observed fraud frequency. Isotonic can create tied scores; Platt is strictly monotone. Scores at 0.50 are not a production decision threshold. Refresh the calibrator only with matured historical labels in controlled updates.


In [5]:
raw_bins = reliability_bins(y_validation, raw_validation)
calibrated_bins = reliability_bins(y_validation, calibrated_validation)
raw_bins.to_csv(ROOT / "reports/probability_calibration_raw_bins.csv", index=False)
calibrated_bins.to_csv(ROOT / "reports/probability_calibration_calibrated_bins.csv", index=False)
display(raw_bins)
display(calibrated_bins)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.plot(raw_bins.mean_probability, raw_bins.fraud_rate, "o-", label="Raw ensemble")
ax.plot(
    calibrated_bins.mean_probability, calibrated_bins.fraud_rate, "o-", label="Platt calibrated"
)
ax.set(
    xlabel="Mean predicted probability",
    ylabel="Observed fraud rate",
    title="External validation reliability",
)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
figure_path = ROOT / "reports/figures/probability_calibration_curve.png"
fig.savefig(figure_path, dpi=160)
plt.close(fig)
print(figure_path)

,bin,count,score_min,score_max,mean_probability,fraud_rate
0,1,8859,6.091649e-07,0.001267,0.000665,0.000452
1,2,8858,1.266697e-03,0.002978,0.002060,0.001693
2,3,8858,2.978243e-03,0.005798,0.004265,0.003613
3,4,8858,5.798476e-03,0.010343,0.007903,0.004741
4,5,8858,1.034418e-02,0.017678,0.013757,0.006999
5,6,8858,1.767944e-02,0.029445,0.023129,0.008241
6,7,8858,2.944580e-02,0.048393,0.038114,0.012305
7,8,8858,4.839826e-02,0.082736,0.063662,0.015353
8,9,8858,8.273850e-02,0.177882,0.120116,0.031045
9,10,8858,1.778912e-01,0.999914,0.459814,0.258975


,bin,count,score_min,score_max,mean_probability,fraud_rate
0,1,8859,0.000005,0.001240,0.000729,0.000452
1,2,8858,0.001240,0.002424,0.001807,0.001693
2,3,8858,0.002424,0.004088,0.003203,0.003613
3,4,8858,0.004088,0.006443,0.005202,0.004741
4,5,8858,0.006443,0.009829,0.008049,0.006999
5,6,8858,0.009830,0.014727,0.012138,0.008241
6,7,8858,0.014727,0.021915,0.018067,0.012305
7,8,8858,0.021916,0.033916,0.027325,0.015353
8,9,8858,0.033917,0.065151,0.046278,0.031045
9,10,8858,0.065154,0.997197,0.243810,0.258975


C:\fraud-detection-system\reports\figures\probability_calibration_curve.png


In [6]:
tracking_uri = f"sqlite:///{(ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-probability-calibration")
tags = {
    "stage": "probability-calibration",
    "dataset": "ieee-cis",
    "selection": "train-oof-temporal",
}
run_ids = {}
for method in ("raw", "platt", "isotonic"):
    rows = selection.loc[selection.method == method].set_index("evaluation_fold")
    with mlflow.start_run(run_name=f"selection-{method}") as run:
        mlflow.set_tags(tags)
        mlflow.log_param("calibration_method", method)
        metrics = {
            f"fold_{fold}_{metric}": float(rows.loc[fold, metric])
            for fold in (2, 3)
            for metric in ("brier", "log_loss", "ece")
        }
        metrics.update(
            {
                f"pooled_{metric}": float(rows.loc["pooled", metric])
                for metric in ("brier", "log_loss", "ece", "pr_auc", "roc_auc")
            }
        )
        mlflow.log_metrics(metrics)
        run_ids[method] = run.info.run_id
with mlflow.start_run(run_name=f"final-{selected}") as run:
    mlflow.set_tags(tags)
    mlflow.log_param("calibration_method", selected)
    mlflow.log_metrics(
        {
            f"validation_{label}_{metric}": float(values[metric])
            for label, values in (("raw", raw_metrics), ("calibrated", calibrated_metrics))
            for metric in ("brier", "log_loss", "ece", "pr_auc", "roc_auc")
        }
    )
    run_ids["final"] = run.info.run_id
print(run_ids)

{'raw': '6984e88918124c27a5c9e82ea98e9b89', 'platt': 'f7fe9f23b08c48978cb729bdecd73eba', 'isotonic': '9763c7c5cd414fa3bb4abe43b5b5f707', 'final': '4d7425b5daf14dc0b3ba6f5741c55397'}


In [7]:
def metric_table(frame):
    columns = list(frame.columns)
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in frame.itertuples(index=False, name=None):
        lines.append(
            "| "
            + " | ".join(
                f"{value:.6f}" if isinstance(value, float) else str(value) for value in row
            )
            + " |"
        )
    return "\n".join(lines)


validation_table = pd.DataFrame(
    [
        {"score": name, **values}
        for name, values in (("raw", raw_metrics), ("calibrated", calibrated_metrics))
    ]
)
report = f"""# Probability calibration summary

Calibration improves probability interpretation; it is not intended to improve ranking PR-AUC.
The raw ensemble overpredicts fraud on the temporal OOF evaluation rows.

## OOF source and integrity

The aligned source is reports/ensemble_oof_predictions.csv: {len(oof):,} unique TRAIN rows, three chronological folds of 41,338 rows, complete binary targets and finite component probabilities. Frozen score = 0.8 LightGBM + 0.2 XGBoost. OOF identifiers belong to TRAIN and do not overlap external VALIDATION. Raw all-fold mean score = {raw_oof.mean():.6f}; prevalence = {oof.y_true.mean():.6f}; range = [{raw_oof.min():.6f}, {raw_oof.max():.6f}].

## Temporal method selection

Fit on Fold 1 then assess Fold 2; fit on Folds 1+2 then assess Fold 3. The raw baseline uses identical evaluation rows. Pooled Brier is primary; Log Loss and ten-bin equal-frequency ECE are secondary. ECE weights absolute bin mean-probability versus fraud-rate gaps by row count. The bins partition all rows, including tied scores. Neither external VALIDATION nor HOLDOUT participates in selection.

{metric_table(selection)}

Selected: **{selected}** by lowest pooled Brier. Platt is simpler and continuous; isotonic introduced score ties and lower ranking sanity metrics. Selected mapping was then fitted on all TRAIN OOF labels and serialized at models/probability_calibrator.joblib. No class weights were used in calibration.

## External VALIDATION

{metric_table(validation_table)}

The rebuilt raw PR-AUC and ROC-AUC match the frozen champion reference to six decimals.

Raw-versus-calibrated reliability bins are stored in reports/probability_calibration_raw_bins.csv and reports/probability_calibration_calibrated_bins.csv. Each bin lists count, score range, mean predicted probability, and observed fraud rate.

{metric_table(calibrated_bins)}

The calibration curve is at reports/figures/probability_calibration_curve.png. Calibrated bin points sit closer to the diagonal overall; the highest risk bin still has residual error. PR-AUC and ROC-AUC are ranking sanity checks, not selection criteria. The Platt transform is monotone, so within-period ranking is preserved apart from numerical precision.

## Limitations and production implications

Ten-bin ECE is a summary sensitive to binning and can conceal within-bin error. Temporal distribution shift remains possible. This is one external VALIDATION assessment and does not establish future reliability. A 0.50 score is not a production threshold; decision threshold design is a later phase. The calibrator must only be refreshed using matured historical labels during controlled retraining/calibration updates.

## Runtime, memory, and tracking

Notebook runtime: {time.perf_counter() - started:.1f} seconds. Prepared matrix estimate: {matrix_mib:.1f} MiB. MLflow experiment: fraud-detection-probability-calibration; tracking URI: {tracking_uri}. Run IDs: {json.dumps(run_ids)}.

HOLDOUT remained untouched: no HOLDOUT access, transformation, prediction, summary, calibration, or evaluation.
"""
(ROOT / "reports/probability_calibration_summary.md").write_text(report, encoding="utf-8")
print("Report written; runtime seconds:", time.perf_counter() - started)

Report written; runtime seconds: 229.73412779999853
